# 03 merge(bf16) + 4bit 양자화 (배포본, 양자화하지 않는 가중치는 fp16)

| 항목 | 값 |
|---|---|
| GPU | A100 40GB 또는 80GB (bf16 14.7B 약 29.5GB가 GPU에 올라가야 함), 고용량 RAM |
| 예상 시간 | 약 40분 (bf16 로드·merge 10분, 로컬 저장 30GB 10분, NF4 재로드·Drive 저장 9.6GB 15분) |
| 예상 컴퓨트 | 약 8~10 CU (A100 약 13 CU/시간 가정. Colab 리소스 창에서 실제 소모율을 확인해 기록) |
| 디스크 | 원본 59GB + merge bf16 30GB (Colab 로컬 디스크, Drive에는 NF4 9.6GB만) |
| 이어 하기 | Drive `merged_nf4/hashes.json`이 있으면 끝난 것이다. 중간에 끊기면 처음부터 다시 한다. |

사전 등록: `docs/spec.md` §6.1~6.3, `docs/decisions.md` 2026-11-10. 설정을 바꾸면 사전 등록 위반이니 멈추고 기록한다.
준비: repo에서 `scripts/pack_distill_bundle.sh`로 만든 `tomato_distill_bundle.tar.gz`를 Drive `MyDrive/tomato-distill/`에 올린다.

In [ ]:
# 공통 준비: Drive 연결, 묶음 풀기, 공용 코드 import
from google.colab import drive
drive.mount("/content/drive")
import os, sys, json, time, gc
DIR = "/content/drive/MyDrive/tomato-distill"
os.environ["TOMATO_DISTILL_DIR"] = DIR
# 묶음은 실행할 때마다 새로 푼다(예전 묶음이 남아 다른 프롬프트로 도는 것을 막는다). sha256을 repo에서 만든 값과 비교한다
import shutil, hashlib
shutil.rmtree(f"{DIR}/bundle", ignore_errors=True)
os.makedirs(f"{DIR}/bundle")
!tar xzf {DIR}/tomato_distill_bundle.tar.gz -C {DIR}/bundle
with open(f"{DIR}/tomato_distill_bundle.tar.gz", "rb") as f:
    print("bundle sha256", hashlib.sha256(f.read()).hexdigest())
sys.path.insert(0, f"{DIR}/bundle/notebooks")
import distill_common as dc
prompt = dc.setup()
from tomato.explain import grounding
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!pip install -q "transformers==5.22.2" "peft==0.21.2" "bitsandbytes==0.50.2" "accelerate==1.15.0"
import importlib.metadata as _md
PINS = {'transformers': '5.22.2', 'peft': '0.21.2', 'bitsandbytes': '0.50.2', 'accelerate': '1.15.0'}
assert {k: _md.version(k) for k in PINS} == PINS, {k: _md.version(k) for k in PINS}
print(PINS)
# Colab에 깔린 torchao 0.10.0을 지운다. peft 0.21.2는 bnb가 아닌 일반 Linear(여기서는 bf16 원본)에 LoRA를 붙일 때
# torchao 판별 함수를 부르고, torchao가 있는데 0.16.0 미만이면 ImportError를 낸다(peft import_utils.is_torchao_available).
# 02는 4bit(bnb) 층이라 bnb 판별에서 먼저 끝나 이 경로를 타지 않았다. torchao는 이 노트북에서 쓰지 않는다.
!pip uninstall -y -q torchao
import importlib; importlib.invalidate_caches()
assert importlib.util.find_spec("torchao") is None, "torchao가 남아 있다"
import torch, shutil
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

## SMOKE: 본 실행 전에 전 과정을 작게 확인
- `SMOKE = True`: 02 SMOKE의 `lora_smoke/chosen` 어댑터로, 학생 원본의 앞 2개 층만 올려 merge → 로컬 저장 → NF4(fp16) 재로드 → Drive 저장 → 해시까지 돈다. 결과는 Drive `merged_nf4_smoke/`(04 SMOKE가 읽음). 본 실행의 `merged_nf4/`는 건드리지 않는다.
- 층을 줄인 것은 시간 때문이다. 호출하는 API와 순서는 본 실행과 같고, 디스크·RAM 용량은 확인하지 못한다. 학생 원본 다운로드(첫 실행 약 15분)는 필요하다.
- 확인이 끝나면 `SMOKE = False`로 바꾸고 처음부터 다시 실행한다.

In [ ]:
SMOKE = True  # 본 실행은 False
ADAPTER = dc.DRIVE / ("lora_smoke" if SMOKE else "lora") / "chosen"
OUT = dc.DRIVE / ("merged_nf4_smoke" if SMOKE else "merged_nf4")
LOCAL = "/content/merged_bf16_smoke" if SMOKE else "/content/merged_bf16"
LAYERS = {"num_hidden_layers": 2} if SMOKE else {}  # SMOKE만: 앞 2개 층만 올린다
print("SMOKE" if SMOKE else "본 실행", ADAPTER, "→", OUT)

In [ ]:
# 1) bf16 원본에 LoRA merge → 로컬 디스크에 저장
if not (OUT / "hashes.json").exists():
    base = AutoModelForCausalLM.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV, dtype=torch.bfloat16, device_map={"": 0}, **LAYERS)
    tok = AutoTokenizer.from_pretrained(dc.STUDENT_ID, revision=dc.STUDENT_REV)
    merged = PeftModel.from_pretrained(base, str(ADAPTER)).merge_and_unload()
    merged.save_pretrained(LOCAL)  # transformers 5.x는 항상 safetensors로 저장한다(safe_serialization 인자 없음)
    tok.save_pretrained(LOCAL)
    del base, merged; gc.collect(); torch.cuda.empty_cache()

In [ ]:
# 2) NF4(이중 양자화)로 다시 읽어 4bit로 저장 → Drive. 데모 서버(2080 Ti)가 bf16을 지원하지 않으므로
#    양자화하지 않는 가중치(embedding·lm_head·norm)와 계산 dtype은 fp16으로 저장한다(spec §6.2)
if not (OUT / "hashes.json").exists():
    q = AutoModelForCausalLM.from_pretrained(LOCAL, quantization_config=dc.bnb_config(dc.EVAL_COMPUTE_DTYPE),
                                             dtype=dc.torch_dtype(dc.EVAL_COMPUTE_DTYPE), device_map={"": 0})
    print("저장할 계산 dtype", q.config.quantization_config.bnb_4bit_compute_dtype, "| 층 수", q.config.num_hidden_layers)
    q.save_pretrained(str(OUT))
    AutoTokenizer.from_pretrained(LOCAL).save_pretrained(str(OUT))
    print("GPU 메모리(4bit 전체 GPU)", round(torch.cuda.memory_allocated() / 1e9, 2), "GB")
    del q; gc.collect(); torch.cuda.empty_cache()
    hashes = dc.dir_hashes(OUT)
    (OUT / "hashes.json").write_text(json.dumps({"source": dc.STUDENT_ID, "revision": dc.STUDENT_REV,
                                                 "adapter": dc.dir_hashes(ADAPTER), "smoke": SMOKE,
                                                 "files": hashes}, indent=1))
print((OUT / "hashes.json").read_text())

## repo로 가져올 파일
- 본 실행(`SMOKE = False`)만: `merged_nf4/hashes.json`만 → `distill/merged_nf4_hashes.json`. 가중치(약 9.6GB)는 Drive에 둔다. `merged_nf4_smoke/`는 가져오지 않는다.

In [ ]:
# 끝: Drive에 다 쓴 뒤 런타임을 끈다(끝난 뒤 GPU가 켜진 채 컴퓨팅 단위를 쓰지 않게).
# 앞 셀에서 오류로 멈추면 이 셀은 실행되지 않는다. 그때는 직접 런타임 > 런타임 연결 해제 및 삭제.
AUTO_SHUTDOWN = True
if AUTO_SHUTDOWN:
    drive.flush_and_unmount()
    from google.colab import runtime
    runtime.unassign()